# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [ ]:
# Imports
import os
import json
from datetime import datetime, timezone
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
# Constants and API clients (OpenRouter for cloud GPT, Ollama for local)
load_dotenv()
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
MODEL_GPT = "openai/gpt-4o-mini"
MODEL_LLAMA = "llama3.2"
OLLAMA_BASE_URL = "http://localhost:11434/v1"

if not OPENROUTER_API_KEY:
    print("Warning: OPENROUTER_API_KEY not set. Add it to a .env file (see .env.example).")

openrouter_client = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=OPENROUTER_API_KEY or "dummy")
ollama_client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

In [ ]:
# System prompts: expertise personas (used as system message)
SYSTEM_PROMPTS = {
    "Technical tutor": (
        "You are a clear technical tutor. Explain code and concepts concisely with examples. "
        "Use markdown for structure. If the user asks for the current time, use the get_current_time tool."
    ),
    "Code reviewer": (
        "You are a senior code reviewer. Explain code snippets, suggest improvements, and point out pitfalls. "
        "Use markdown. If the user asks what time it is, use the get_current_time tool."
    ),
    "LLM explainer": (
        "You explain how LLMs, APIs and prompt engineering work. Be precise and educational. "
        "Use markdown. If the user asks for the current time, use the get_current_time tool."
    ),
}

In [ ]:
# Tool: get current time (demonstrates tool use — ask "What time is it?" in the chat)
def get_current_time(timezone_name: str = "UTC") -> str:
    """Return the current date and time in the given timezone (e.g. UTC, Europe/Paris)."""
    try:
        from zoneinfo import ZoneInfo
        tz = ZoneInfo(timezone_name)
    except Exception:
        tz = timezone.utc
    now = datetime.now(tz)
    return f"Current time in {timezone_name}: {now.strftime('%Y-%m-%d %H:%M:%S %Z')}"

get_current_time_tool = {
    "type": "function",
    "function": {
        "name": "get_current_time",
        "description": "Get the current date and time in a given timezone (e.g. UTC, Europe/Paris, America/New_York).",
        "parameters": {
            "type": "object",
            "properties": {
                "timezone_name": {
                    "type": "string",
                    "description": "IANA timezone name, e.g. UTC or Europe/Paris",
                    "default": "UTC",
                },
            },
            "required": [],
            "additionalProperties": False,
        },
    },
}
TOOLS = [get_current_time_tool]

def handle_tool_calls(message):
    """Execute tool calls and return tool results for the API."""
    results = []
    for tc in message.tool_calls:
        name = tc.function.name
        args = json.loads(tc.function.arguments) if tc.function.arguments else {}
        if name == "get_current_time":
            out = get_current_time(**args)
        else:
            out = f"Unknown tool: {name}"
        results.append({"role": "tool", "content": out, "tool_call_id": tc.id})
    return results

In [ ]:
# Streaming chat: supports GPT (OpenRouter) with tool use, or Ollama Llama
def _ollama_stream(client, model, messages):
    """Stream from Ollama. Yields cumulative text."""
    acc = ""
    try:
        stream = client.chat.completions.create(model=model, messages=messages, stream=True)
        for chunk in stream:
            delta = chunk.choices[0].delta.content or ""
            acc += delta
            yield acc
    except Exception as e:
        yield f"Ollama error. Run `ollama serve` and `ollama pull {model}`. {e}"

def _gpt_stream(client, model, messages):
    """Stream from GPT via OpenRouter. Handles tool calls; streams text when no tools used."""
    response = client.chat.completions.create(model=model, messages=messages, tools=TOOLS)
    while response.choices[0].finish_reason == "tool_calls":
        msg = response.choices[0].message
        tool_calls = [{"id": tc.id, "type": "function", "function": {"name": tc.function.name, "arguments": tc.function.arguments or "{}"}} for tc in msg.tool_calls]
        messages.append({"role": "assistant", "content": msg.content or "", "tool_calls": tool_calls})
        messages.extend(handle_tool_calls(msg))
        response = client.chat.completions.create(model=model, messages=messages, tools=TOOLS)
    text = response.choices[0].message.content or ""
    yield text

def chat_stream(message, history, model_choice, persona_key):
    """Yields cumulative response for Gradio streaming. history = list of {role, content}."""
    messages = [{"role": h["role"], "content": str(h.get("content", ""))} for h in history]
    system = SYSTEM_PROMPTS.get(persona_key, list(SYSTEM_PROMPTS.values())[0])
    messages = [{"role": "system", "content": system}] + messages + [{"role": "user", "content": message}]

    use_ollama = "Ollama" in model_choice
    client = ollama_client if use_ollama else openrouter_client
    model = MODEL_LLAMA if use_ollama else MODEL_GPT

    if use_ollama:
        yield from _ollama_stream(client, model, messages)
    else:
        yield from _gpt_stream(client, model, messages)

In [ ]:
# Gradio UI: model switch, persona (system prompt), streaming chat
with gr.Blocks(title="Technical Q&A — Week 2", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        """
        ## Technical Q&A prototype (Week 2)
        - **Model:** OpenRouter GPT or Ollama Llama
        - **Persona:** system prompt sets expertise (tutor, code reviewer, LLM explainer)
        - **Streaming** answers; **tool:** ask *What time is it?* to see the assistant use the clock.
        """
    )
    with gr.Row():
        model_choice = gr.Dropdown(
            ["OpenRouter GPT", "Ollama Llama"],
            value="OpenRouter GPT",
            label="Model",
        )
        persona = gr.Dropdown(
            list(SYSTEM_PROMPTS.keys()),
            value="Technical tutor",
            label="Persona",
        )
    chatbot = gr.Chatbot(height=400, type="messages", label="Chat")
    msg = gr.Textbox(placeholder="Ask a technical question or 'What time is it?'", label="Message")
    send = gr.Button("Send", variant="primary")
    clear = gr.Button("Clear")

    def respond(message, history, model, persona_name):
        if not message or not message.strip():
            return "", history
        history = history + [{"role": "user", "content": message}]
        full = ""
        for chunk in chat_stream(message, history[:-1], model, persona_name):
            full = chunk
            yield "", history + [{"role": "assistant", "content": full}]

    msg.submit(respond, [msg, chatbot, model_choice, persona], [msg, chatbot])
    send.click(respond, [msg, chatbot, model_choice, persona], [msg, chatbot])
    clear.click(lambda: [], None, chatbot, queue=False)

demo.launch(inbrowser=True)